In [ ]:
import os
from io import BytesIO
from pathlib import Path

import boto3
import dotenv
import pandas as pd
import altair as alt

In [ ]:
env_path = dotenv.find_dotenv(usecwd=True)
dotenv.load_dotenv(env_path)

In [ ]:
COUNTY_LIST: set[str] = {
    "Romania",
    "Hungary",
    "Poland"
}
VARIABLE: str = "t2m_max"

In [ ]:
def get_era5_country_averages() -> pd.DataFrame:
    s3 = boto3.client("s3")
    bucket: str = os.getenv("S3_BUCKET_NAME")
    assert bucket
    response = s3.get_object(Bucket=bucket, Key="analysis/daily-country-averages/era5.parquet")
    daily = pd.read_parquet(
        BytesIO(response["Body"].read()),
        columns=["country", "date", VARIABLE],
    )
    daily["date"] = pd.to_datetime(daily["date"])
    daily["year"] = daily["date"].dt.year
    daily = daily[daily["country"].isin(COUNTY_LIST)]
    return daily

In [ ]:
daily_df: pd.DataFrame = get_era5_country_averages()

In [ ]:
daily_df.head()

In [ ]:
baseline_df: pd.DataFrame = daily_df.loc[
    (daily_df["year"].between(1961, 1990)) &
    (daily_df["date"].dt.month.isin([6, 7]))
].copy()

In [ ]:
baseline_df.head()

In [ ]:
# Make sure there are 30 unique years
assert baseline_df["year"].nunique() == 30

In [ ]:
# Full date coverage for the baseline period
assert baseline_df["date"].nunique() == 61 * 30

In [ ]:
# No missing values
assert baseline_df.isna().sum().sum() == 0

In [ ]:
baseline_mean: pd.DataFrame = (
    baseline_df.groupby("country")[VARIABLE]
        .mean()
        .reset_index()
)

In [ ]:
baseline_mean.head()

In [ ]:
analysis_df: pd.DataFrame = (
    daily_df.loc[
        (daily_df["year"].between(1961, 2026)) &
        (daily_df["date"].dt.month.isin([6, 7]))
    ].copy()
     .groupby(["country", "year"])[VARIABLE]
     .mean()
     .reset_index()
     .merge(baseline_mean, on=["country"], how="inner", validate="many_to_one", suffixes=("", "_baseline"))
     .assign(anomaly=lambda x: x[VARIABLE] - x[f"{VARIABLE}_baseline"])
)

In [ ]:
analysis_df.head()

In [ ]:
alt.Chart(analysis_df).mark_bar().encode(
    x=alt.X(
        'year:Q',
        title='Year',
        axis=alt.Axis(
            format="d",
            tickMinStep=5,
        ),
    ),
    y='anomaly:Q',
    facet='country:N',
    color='country:N'
 ).properties(width=300)

In [ ]:
pivot_df: pd.DataFrame = analysis_df.pivot(
    index='year',
    columns='country',
    values='anomaly'
).round(3)

In [ ]:
pivot_df.tail(10)

In [ ]:
pivot_df.tail(10).mean()

Over the past decade, Hungary's June and July highs have exceeded the 1961-1990 average by 3.1 degrees Celsius (5.6 Fahrenheit), according to the Reuters Climate Monitor. Romania isn't far behind, with an averaged 3 degrees Celsius (5.4 Fahrenheit) above normal.

In [ ]:
pivot_df.to_csv("pivot.csv", index=True)